# Deep heads on a regularised tree encoding — MagicTelescope (Kaggle)

Same experiment as the Colab notebook, rebuilt for Kaggle's **Save & Run All**. It runs start to finish with no clicks: no Drive, no upload dialogs, nothing that waits for input.

### Before you press Save & Run All
In the right-hand **Settings** panel:
1. **Accelerator → GPU T4 x2** (one GPU is used; the T4 is what the timings assume)
2. **Internet → On** (needed to fetch the code and the dataset)

Both need a phone-verified Kaggle account. Then **Save Version → Save & Run All (Commit)**.

### Where the results appear
Everything is written to `/kaggle/working`, which Kaggle keeps as the notebook's **Output** when the run finishes. Open the saved version and go to the **Output** section to download `deep_head_<date>.zip`: every run's training log, the result files with full epoch histories, the summary tables, an Excel workbook, the three charts and a README.

### Time
About **1.5 to 2.5 hours** on a T4 for all 72 runs, well inside Kaggle's 12-hour limit for one run. Cell 5 measures the transformer's real speed on your GPU and prints the projection into the log.

### If a run ever fails partway
Kaggle saves the output of a failed version too, including mid-training checkpoints. To continue: open a new version of this notebook, **Add Input → Your Work → this notebook's previous version**, then Save & Run All again. Cell 2 finds the attached output and copies it back in; finished runs are skipped and an interrupted run continues from its last checkpoint.

### What runs
| part | runs | what |
|---|---|---|
| A | 2 | linear reference, penalty chosen on validation, on both encodings |
| B | 54 | MLP, TabResNet and the bit-level transformer, 3 sizes x 6 regularisation settings, shallow bits |
| C | 16 | the same heads on the full encoding, 2 settings: capacity versus generalisation |

Regularisation settings: none, hidden dropout 0.3, weight decay 0.3, input dropout 0.3, input dropout 0.5, and all three together. Input dropout drops whole bits before the head sees them, which is the only setting that reaches the first layer, where most of the parameters and most of the memorising sit.

In [ ]:
# 1 · check the GPU and the internet connection
import torch, subprocess, time
T0 = time.time()
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| count:', torch.cuda.device_count())
else:
    print('NO GPU — set Settings > Accelerator > GPU T4 x2 and save again')
net = subprocess.run(['curl', '-sI', '--max-time', '10', 'https://github.com'],
                     capture_output=True, text=True).returncode == 0
print('internet:', 'on' if net else 'OFF — set Settings > Internet > On and save again')

In [ ]:
# 2 · lay out the output folders, and restore anything a previous version left behind
import os, glob, shutil, tarfile
WORK  = '/kaggle/working'
ROOT  = f'{WORK}/deep_head'   # results: kept as the notebook's Output
CACHE = f'{WORK}/cache'       # the forest and the bits, so a rerun skips them
CKPT  = f'{WORK}/ckpt'        # mid-training checkpoints, every 10 epochs
for d in (ROOT, CACHE, CKPT):
    os.makedirs(d, exist_ok=True)

# A previous version's output, attached via Add Input, appears under /kaggle/input.
restored = 0
for prev in glob.glob('/kaggle/input/**/deep_head', recursive=True):
    if not os.path.isdir(prev):
        continue
    shutil.copytree(prev, ROOT, dirs_exist_ok=True)
    base = os.path.dirname(prev)
    for name, dst in (('cache', CACHE), ('ckpt', CKPT)):
        if os.path.isdir(f'{base}/{name}'):
            shutil.copytree(f'{base}/{name}', dst, dirs_exist_ok=True)
    restored = len(glob.glob(f'{ROOT}/*/*/deephead_*.json'))
    print(f'restored a previous version from {prev}: {restored} finished runs, '
          f'{len(glob.glob(CKPT + "/ck_*.pt"))} checkpoints')
if not restored:
    print('no previous version attached — starting fresh')

# Fallback if OpenML is down: attach a dataset holding openml_cache_clean5.tar.gz
bundles = glob.glob('/kaggle/input/**/openml_cache_*.tar.gz', recursive=True)
if bundles:
    dst = '/root/.cache/openml/org/openml/www'
    os.makedirs(dst, exist_ok=True)
    with tarfile.open(bundles[0]) as t:
        t.extractall(dst)
    print('OpenML cache bundle extracted from', bundles[0])

In [ ]:
# 3 · get the code (outside /kaggle/working, so the repo is not saved as output)
!rm -rf /tmp/TKCE && git clone -q https://github.com/sushanedulloo/TKCE.git /tmp/TKCE
%cd /tmp/TKCE
!git log -1 --format='code version: %h  %s'
!pip install -q openml 2>&1 | grep -v 'already satisfied' | tail -2
SUITE = f'--root {ROOT} --cache-dir {CACHE} --ckpt-dir {CKPT}'
print(SUITE)

## Part A — the linear reference

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part A 2>&1 | tee -a {ROOT}/suite.log
print(f'elapsed {(time.time()-T0)/60:.1f} min')

In [ ]:
# 5 · how fast is the bit-level transformer on this GPU? (3 epochs, then a projection)
!python -u run_deep_head.py --task 361065 --drop-layers 4,5 --head bittf --size large \
    --batch-size 64 --epochs 150 --time-probe 3 --cache-dir {CACHE} --out /tmp/probe \
    2>&1 | grep -E '^\[bits\]|^\[head\]|^\[probe\]'

## Part B — deep heads on the shallow encoding

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part B 2>&1 | tee -a {ROOT}/suite.log
print(f'elapsed {(time.time()-T0)/60:.1f} min')

## Part C — the same heads on the full encoding

In [ ]:
!python -u run_deep_head_suite.py {SUITE} --part C 2>&1 | tee -a {ROOT}/suite.log
print(f'elapsed {(time.time()-T0)/60:.1f} min')

In [ ]:
# 6 · summary: best run per head, chosen on validation, never on test
#     wrapped so a problem here can never stop the bundle cell from running
!python -u run_deep_head_suite.py {SUITE} --aggregate-only 2>&1 | tail -20
try:
    import pandas as pd
    pd.set_option('display.width', 250)
    print(pd.read_csv(f'{ROOT}/suite_summary.csv').to_string(index=False))
except Exception as e:
    print('summary unavailable:', e)

In [ ]:
# 7 · did any deep head beat the linear one, and what did regularisation do?
try:
    t = pd.read_csv(f'{ROOT}/suite_long.csv')
    lin = t[(t['head'] == 'linear') & (t.bits == 'shallow')].test.iloc[0]
    ceil = t.tree_ceiling.iloc[0]
    sh = t[(t.bits == 'shallow') & (t['head'] != 'linear')]
    print(f'linear head {lin:.4f}  |  best tree {ceil:.4f}  |  '
          f'scores are uncertain by about 0.01 on this test set\n')
    beat = sh[sh.test > lin + 0.01]
    print(f'deep runs clearly beating the linear head: {len(beat)} of {len(sh)}\n')
    for h in ['mlp', 'tabresnet', 'bittf']:
        g = sh[sh['head'] == h]
        if not len(g):
            continue
        b = g.sort_values('val').iloc[-1]
        print(f'{h:10s} best by validation: test {b.test:.4f} ({b["size"]}, {b.reg}), '
              f'gap {b.gap:+.4f}, vs linear {b.test-lin:+.4f}, vs tree {b.test-ceil:+.4f}')
        for reg, gg in g.groupby('reg'):
            print(f'    {reg:8s} mean test {gg.test.mean():.4f}   mean gap {gg.gap.mean():.4f}'
                  f'   mean log loss {gg.test_logloss.mean():.4f}')
except Exception as e:
    print('analysis unavailable:', e)

In [ ]:
# 8 · the charts
try:
    from IPython.display import Image, display
    for f in ['fig1_heads.png', 'fig2_capacity.png', 'fig3_curves.png']:
        p = f'{ROOT}/{f}'
        if os.path.exists(p):
            print('=' * 30, f, '=' * 30)
            display(Image(p))
except Exception as e:
    print('charts unavailable:', e)

In [ ]:
# 9 · BUNDLE everything into /kaggle/working for download from the Output section
!python -u run_deep_head_suite.py {SUITE} --aggregate-only --bundle 2>&1 | grep -E 'bundle ->|nothing'
zips = sorted(glob.glob(f'{WORK}/deep_head_*.zip'), key=os.path.getmtime)
if zips:
    z = zips[-1]
    print(f'\nready to download: {os.path.basename(z)}  ({os.path.getsize(z)/1e6:.1f} MB)')
    print('open this saved version and go to the Output section to download it')
    !unzip -l {z} | tail -n +4 | head -40
else:
    print('no bundle was written; the raw results are still in', ROOT)
left = glob.glob(f'{CKPT}/ck_*.pt')
if left:
    print(f'\n{len(left)} run(s) were interrupted mid-training; their checkpoints are saved '
          f'in the output, so attach this version as input and run again to finish them')
print(f'\ntotal time {(time.time()-T0)/60:.1f} min')